In [1]:
from kaggle.api.kaggle_api_extended import KaggleApi

In [2]:
api = KaggleApi()
api.authenticate()

api.dataset_download_files(
    'rohanrao/formula-1-world-championship-1950-2020',
    path='./f1_datasets',
    unzip=True
)

Dataset URL: https://www.kaggle.com/datasets/rohanrao/formula-1-world-championship-1950-2020


In [3]:
import pandas as pd
import os
# Load the datasets

datasets = ['circuits', 'constructors', 'drivers', 'races', 'results']

dataframes = {}
for dataset in datasets:
    file_path = os.path.join('./f1_datasets', f'{dataset}.csv')
    dataframes[dataset] = pd.read_csv(file_path)

dataframes

{'circuits':     circuitId   circuitRef                                  name  \
 0           1  albert_park        Albert Park Grand Prix Circuit   
 1           2       sepang          Sepang International Circuit   
 2           3      bahrain         Bahrain International Circuit   
 3           4    catalunya        Circuit de Barcelona-Catalunya   
 4           5     istanbul                         Istanbul Park   
 ..        ...          ...                                   ...   
 72         75     portimao    Autódromo Internacional do Algarve   
 73         76      mugello  Autodromo Internazionale del Mugello   
 74         77       jeddah               Jeddah Corniche Circuit   
 75         78       losail          Losail International Circuit   
 76         79        miami         Miami International Autodrome   
 
         location       country       lat        lng  alt  \
 0      Melbourne     Australia -37.84970  144.96800   10   
 1   Kuala Lumpur      Malaysia   2.

In [4]:
dataframes['races'].columns, dataframes['results'].columns, dataframes['constructors'].columns, dataframes['circuits'].columns, dataframes['drivers'].columns

(Index(['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time', 'url',
        'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time',
        'quali_date', 'quali_time', 'sprint_date', 'sprint_time'],
       dtype='object'),
 Index(['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid',
        'position', 'positionText', 'positionOrder', 'points', 'laps', 'time',
        'milliseconds', 'fastestLap', 'rank', 'fastestLapTime',
        'fastestLapSpeed', 'statusId'],
       dtype='object'),
 Index(['constructorId', 'constructorRef', 'name', 'nationality', 'url'], dtype='object'),
 Index(['circuitId', 'circuitRef', 'name', 'location', 'country', 'lat', 'lng',
        'alt', 'url'],
       dtype='object'),
 Index(['driverId', 'driverRef', 'number', 'code', 'forename', 'surname', 'dob',
        'nationality', 'url'],
       dtype='object'))

In [5]:
dataframes['races'].drop(columns=['url', 'round', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time',
        'quali_date', 'quali_time', 'sprint_date', 'sprint_time', 'time'], inplace=True)
dataframes['races'].rename(columns={'name': 'GP name'}, inplace=True)
dataframes['constructors'].drop(columns=['constructorRef', 'url'], inplace=True)
dataframes['constructors'].rename(columns={'name': 'team', 'nationality': 'team_home'}, inplace=True)
dataframes['drivers'].drop(columns=['forename', 'surname', 'code', 'number', 'url'], inplace=True)
dataframes['drivers'].rename(columns={'name': 'driver', 'nationality': 'driver_home'}, inplace=True)
dataframes['results'].drop(columns=['positionText', 'points', 'laps', 'time', 'milliseconds', 'positionOrder', 'fastestLapTime',
                                    'fastestLapSpeed', 'rank', 'laps', 'time', 'number'], inplace=True)
dataframes['results'].rename(columns={'position': 'finish', 'grid': 'start'}, inplace=True)
dataframes['circuits'].drop(columns=['circuitRef', 'url', 'lat', 'lng', 'alt'], inplace=True)
dataframes['circuits'].rename(columns={'name': 'circuit name'}, inplace=True)

In [6]:
f1_data = dataframes['races'].merge(
    dataframes['results'], on='raceId', how='left'
    ).merge(
        dataframes['constructors'], on='constructorId', how='left'
    ).merge(
        dataframes['drivers'], on='driverId', how='left'
    ).merge(
        dataframes['circuits'], on='circuitId', how='left'
        )



In [7]:
f1_data = f1_data[f1_data['year'] >= 2012]
f1_data.shape

(5524, 20)

In [8]:
f1_data['date'] = pd.to_datetime(f1_data['date'])
f1_data['dob'] = pd.to_datetime(f1_data['dob'])

In [9]:
f1_data.columns

Index(['raceId', 'year', 'circuitId', 'GP name', 'date', 'resultId',
       'driverId', 'constructorId', 'start', 'finish', 'fastestLap',
       'statusId', 'team', 'team_home', 'driverRef', 'dob', 'driver_home',
       'circuit name', 'location', 'country'],
      dtype='object')

In [10]:
f1_data.drop(columns=['resultId', 'constructorId', 'fastestLap', 'driverId', 'circuitId', 'raceId'], inplace=True)
f1_data.shape

(5524, 14)

In [11]:
f1_data

,year,GP name,date,start,finish,statusId,team,team_home,driverRef,dob,driver_home,circuit name,location,country
21235,2012,Australian Grand Prix,2012-03-18,2,1,1,McLaren,British,button,1980-01-19,British,Albert Park Grand Prix Circuit,Melbourne,Australia
21236,2012,Australian Grand Prix,2012-03-18,6,2,1,Red Bull,Austrian,vettel,1987-07-03,German,Albert Park Grand Prix Circuit,Melbourne,Australia
21237,2012,Australian Grand Prix,2012-03-18,1,3,1,McLaren,British,hamilton,1985-01-07,British,Albert Park Grand Prix Circuit,Melbourne,Australia
21238,2012,Australian Grand Prix,2012-03-18,5,4,1,Red Bull,Austrian,webber,1976-08-27,Australian,Albert Park Grand Prix Circuit,Melbourne,Australia
21239,2012,Australian Grand Prix,2012-03-18,12,5,1,Ferrari,Italian,alonso,1981-07-29,Spanish,Albert Park Grand Prix Circuit,Melbourne,Australia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26754,2024,Abu Dhabi Grand Prix,2024-12-08,14,16,11,Haas F1 Team,American,kevin_magnussen,1992-10-05,Danish,Yas Marina Circuit,Abu Dhabi,UAE
26755,2024,Abu Dhabi Grand Prix,2024-12-08,12,17,5,RB F1 Team,Italian,lawson,2002-02-11,New Zealander,Yas Marina Circuit,Abu Dhabi,UAE
26756,2024,Abu Dhabi Grand Prix,2024-12-08,9,\N,130,Sauber,Swiss,bottas,1989-08-28,Finnish,Yas Marina Circuit,Abu Dhabi,UAE
26757,2024,Abu Dhabi Grand Prix,2024-12-08,20,\N,5,Williams,British,colapinto,2003-05-27,Argentinian,Yas Marina Circuit,Abu Dhabi,UAE


In [12]:
print(f1_data['team_home'].unique())
print(f1_data['driver_home'].unique())
print(f1_data['country'].unique())

['British' 'Austrian' 'Italian' 'Swiss' 'Indian' 'German' 'Russian'
 'Malaysian' 'Spanish' 'American' 'French']
['British' 'German' 'Australian' 'Spanish' 'Japanese' 'Finnish' 'Mexican'
 'French' 'Venezuelan' 'Brazilian' 'Russian' 'Indian' 'Belgian' 'Dutch'
 'Danish' 'Swedish' 'American' 'Indonesian' 'Italian' 'Canadian'
 'New Zealander' 'Monegasque' 'Thai' 'Polish' 'Chinese' 'Argentinian ']
['Australia' 'Malaysia' 'China' 'Bahrain' 'Spain' 'Monaco' 'Canada' 'UK'
 'Germany' 'Hungary' 'Belgium' 'Italy' 'Singapore' 'Japan' 'Korea' 'India'
 'UAE' 'USA' 'Brazil' 'Austria' 'Russia' 'Mexico' 'Azerbaijan' 'France'
 'Portugal' 'Turkey' 'Qatar' 'Netherlands' 'Saudi Arabia' 'United States']


In [13]:
def nationality(x):
    x = str(x).strip().lower()
    if x in ['austrian', 'austria']:
        return 'AUT'
    if x in ['australian', 'australia']:
        return 'AUS'
    if x in ['indian', 'india']:
        return 'IND'
    if x in ['indonesian', 'indonesia']:
        return 'INA'
    if x in ['uk', 'british', 'england', 'great britain']:
        return 'BRI'
    if x in ['usa', 'united states', 'american']:
        return 'AME'
    if x in ['fra', 'france', 'french']:
        return 'FRE'
    return x[:3].upper()

f1_data['driver_home'] = f1_data['driver_home'].apply(nationality)
f1_data['team_home'] = f1_data['team_home'].apply(nationality)
f1_data['country'] = f1_data['country'].apply(nationality)

f1_data['driver home'] = (f1_data['driver_home'] == f1_data['country']).astype(int)
f1_data['team home'] = (f1_data['team_home'] == f1_data['country']).astype(int)

In [14]:
print(f1_data['driver home'].unique())
print(f1_data['team home'].unique())
f1_data = f1_data.drop(columns=['driver_home', 'team_home'])
f1_data

[0 1]
[0 1]


,year,GP name,date,start,finish,statusId,team,driverRef,dob,circuit name,location,country,driver home,team home
21235,2012,Australian Grand Prix,2012-03-18,2,1,1,McLaren,button,1980-01-19,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
21236,2012,Australian Grand Prix,2012-03-18,6,2,1,Red Bull,vettel,1987-07-03,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
21237,2012,Australian Grand Prix,2012-03-18,1,3,1,McLaren,hamilton,1985-01-07,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
21238,2012,Australian Grand Prix,2012-03-18,5,4,1,Red Bull,webber,1976-08-27,Albert Park Grand Prix Circuit,Melbourne,AUS,1,0
21239,2012,Australian Grand Prix,2012-03-18,12,5,1,Ferrari,alonso,1981-07-29,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26754,2024,Abu Dhabi Grand Prix,2024-12-08,14,16,11,Haas F1 Team,kevin_magnussen,1992-10-05,Yas Marina Circuit,Abu Dhabi,UAE,0,0
26755,2024,Abu Dhabi Grand Prix,2024-12-08,12,17,5,RB F1 Team,lawson,2002-02-11,Yas Marina Circuit,Abu Dhabi,UAE,0,0
26756,2024,Abu Dhabi Grand Prix,2024-12-08,9,\N,130,Sauber,bottas,1989-08-28,Yas Marina Circuit,Abu Dhabi,UAE,0,0
26757,2024,Abu Dhabi Grand Prix,2024-12-08,20,\N,5,Williams,colapinto,2003-05-27,Yas Marina Circuit,Abu Dhabi,UAE,0,0


In [ ]:
f1_data = f1_data.reset_index(drop=True)
f1_data

,year,GP name,date,start,finish,statusId,team,driverRef,dob,circuit name,location,country,driver home,team home
0,2012,Australian Grand Prix,2012-03-18,2,1,1,McLaren,button,1980-01-19,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
1,2012,Australian Grand Prix,2012-03-18,6,2,1,Red Bull,vettel,1987-07-03,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
2,2012,Australian Grand Prix,2012-03-18,1,3,1,McLaren,hamilton,1985-01-07,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
3,2012,Australian Grand Prix,2012-03-18,5,4,1,Red Bull,webber,1976-08-27,Albert Park Grand Prix Circuit,Melbourne,AUS,1,0
4,2012,Australian Grand Prix,2012-03-18,12,5,1,Ferrari,alonso,1981-07-29,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5519,2024,Abu Dhabi Grand Prix,2024-12-08,14,16,11,Haas F1 Team,kevin_magnussen,1992-10-05,Yas Marina Circuit,Abu Dhabi,UAE,0,0
5520,2024,Abu Dhabi Grand Prix,2024-12-08,12,17,5,RB F1 Team,lawson,2002-02-11,Yas Marina Circuit,Abu Dhabi,UAE,0,0
5521,2024,Abu Dhabi Grand Prix,2024-12-08,9,\N,130,Sauber,bottas,1989-08-28,Yas Marina Circuit,Abu Dhabi,UAE,0,0
5522,2024,Abu Dhabi Grand Prix,2024-12-08,20,\N,5,Williams,colapinto,2003-05-27,Yas Marina Circuit,Abu Dhabi,UAE,0,0


In [17]:
missing = f1_data[f1_data['dob'].isna()]
missing['driverRef'].unique()

array([], dtype=object)

In [18]:
season2025 = pd.read_csv('./datasets/season2025_results.csv')

In [19]:
f1_data = pd.concat([f1_data, season2025])
f1_data.reset_index(drop=True)
f1_data

,year,GP name,date,start,finish,statusId,team,driverRef,dob,circuit name,location,country,driver home,team home
21235,2012,Australian Grand Prix,2012-03-18 00:00:00,2,1,1,McLaren,button,1980-01-19 00:00:00,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
21236,2012,Australian Grand Prix,2012-03-18 00:00:00,6,2,1,Red Bull,vettel,1987-07-03 00:00:00,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
21237,2012,Australian Grand Prix,2012-03-18 00:00:00,1,3,1,McLaren,hamilton,1985-01-07 00:00:00,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
21238,2012,Australian Grand Prix,2012-03-18 00:00:00,5,4,1,Red Bull,webber,1976-08-27 00:00:00,Albert Park Grand Prix Circuit,Melbourne,AUS,1,0
21239,2012,Australian Grand Prix,2012-03-18 00:00:00,12,5,1,Ferrari,alonso,1981-07-29 00:00:00,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233,2025,Austrian Grand Prix,2025-06-29,18,16,1,Red Bull,tsunoda,2000-05-11,Red Bull Ring,Spielberg,AUT,0,1
234,2025,Austrian Grand Prix,2025-06-29,12,17,3,Williams,albon,1996-03-23,Red Bull Ring,Spielberg,AUT,0,0
235,2025,Austrian Grand Prix,2025-06-29,7,18,3,Red Bull,max_verstappen,1997-09-30,Red Bull Ring,Spielberg,AUT,0,1
236,2025,Austrian Grand Prix,2025-06-29,9,19,3,Mercedes,antonelli,NaN,Red Bull Ring,Spielberg,AUT,0,0


In [20]:
f1_data = f1_data.rename(columns={
    'driverRef': 'driver',
})

In [21]:
f1_data['driver_dnf'] = f1_data['statusId'].apply(lambda x: 1 if x in [3,4,20,29,31,41,68,73,81,97,82,104,107,130,137] else 0)
f1_data['constructor_dnf'] = f1_data['statusId'].apply(lambda x: 1 if x not in [3,4,20,29,31,41,68,73,81,97,82,104,107,130,137,1] else 0)

In [22]:
missing = f1_data[f1_data['dob'].isna()]
missing['driver'].unique()

array(['antonelli', 'bortoleto', 'hadjar'], dtype=object)

In [24]:
manual_dobs = {
    'bortoleto': '2004-10-14',
    'hadjar'   : '2004-09-28',
    'antonelli': '2006-08-25'
}

for ref, date_str in manual_dobs.items():
    f1_data.loc[
        (f1_data['driver'] == ref) & (f1_data['dob'].isna()),
        'dob'
    ] = pd.to_datetime(date_str)


In [35]:
f1_data['date'] = pd.to_datetime(f1_data['date'])
f1_data['dob'] = pd.to_datetime(f1_data['dob'])
f1_data.dtypes

year                        int64
GP name                    object
date               datetime64[ns]
start                       int64
finish                     object
statusId                    int64
team                       object
driver                     object
dob                datetime64[ns]
circuit name               object
location                   object
country                    object
driver home                 int64
team home                   int64
driver_dnf                  int64
constructor_dnf             int64
dtype: object

In [36]:
f1_data['age_at_gp_in_days'] = abs(f1_data['dob']-f1_data['date'])
f1_data['age_at_gp_in_days'] = f1_data['age_at_gp_in_days'].apply(lambda x: str(x).split(' ')[0])

In [37]:
f1_data['team'].unique()

array(['McLaren', 'Red Bull', 'Ferrari', 'Sauber', 'Lotus F1',
       'Toro Rosso', 'Force India', 'Mercedes', 'Williams', 'Marussia',
       'Caterham', 'HRT', 'Manor Marussia', 'Haas F1 Team', 'Renault',
       'Alfa Romeo', 'Racing Point', 'AlphaTauri', 'Aston Martin',
       'Alpine F1 Team', 'RB F1 Team', 'Alpine'], dtype=object)

In [38]:
f1_data['team'] = f1_data['team'].apply(lambda x: 'Aston Martin' if x=='Force India' else x)
f1_data['team'] = f1_data['team'].apply(lambda x: 'Aston Martin' if x=='Racing Point' else x)
f1_data['team'] = f1_data['team'].apply(lambda x: 'Sauber' if x=='Alfa Romeo' else x)
f1_data['team'] = f1_data['team'].apply(lambda x: 'Renault' if x=='Lotus F1' else x)
f1_data['constructor'] = f1_data['team'].apply(lambda x: 'Renault' if x=='Alpine' else x)
f1_data['team'] = f1_data['team'].apply(lambda x: 'AlphaTauri' if x=='Toro Rosso' else x)

In [ ]:
dnf_by_driver = f1_data.groupby('driver').sum()['driver_dnf']
driver_race_entered = f1_data.groupby('driver').count()['driver_dnf']
driver_dnf_ratio = (dnf_by_driver/driver_race_entered)
driver_confidence = 1-driver_dnf_ratio
driver_confidence_dict = dict(zip(driver_confidence.index,driver_confidence))


dnf_by_constructor = f1_data.groupby('team').sum()['constructor_dnf']
constructor_race_entered = f1_data.groupby('team').count()['constructor_dnf']
constructor_dnf_ratio = (dnf_by_constructor/constructor_race_entered)
constructor_relaiblity = 1-constructor_dnf_ratio
constructor_relaiblity_dict = dict(zip(constructor_relaiblity.index,constructor_relaiblity))